# Nepali Lemmatization

### Define Lexical Analyser

In [1]:
import joblib
from sklearn.pipeline import Pipeline
from nepali_pipeline import NepaliLexicalAnalyzer, NEPALI_STOP_WORDS

### Prepare Data using pre-defined dictionary
#### Use heuristic to generate training dataset

In [2]:
from nepali_pipeline import NepaliHeuristicDatasetGenerator

### The HMM Morphological Analyzer (Lemmatizer):

In [3]:
import re
from pathlib import Path
from nepali_pipeline import NepaliHMMLemmatizer, NepaliStopWordRemover

# --- Execution ---

DATA_DIR = Path("data")

# A handful of files in the corpus contain stray non-UTF-8 bytes (common in
# scraped web text); errors="replace" keeps the rest of the file intact
# instead of failing the whole read.
sentences = [
    path.read_text(encoding="utf-8-sig", errors="replace").replace("﻿", "").strip()
    for path in sorted(DATA_DIR.glob("*.txt"))
]

# Extract tokens (X) and target lemmas (y) from raw sentences using generator
X_train_words, y_train_lemmas = NepaliHeuristicDatasetGenerator().transform(sentences)

# Flatten sentence groups to align with raw sentence lists for Pipeline fitting.
# Must apply the same "has at least one word token" filter as
# NepaliHeuristicDatasetGenerator.transform, or punctuation-only fragments (common
# in real article text) desync X_train_raw from y_train_lemmas.
TOKEN_RE = re.compile(r'[^\s।,\.!?;:()\'"“”]+')
X_train_raw = [
    s.strip()
    for essay in sentences
    for s in re.split(r'[।\?\!]', essay)
    if s.strip() and TOKEN_RE.findall(s.strip())
]

# Build full Pipeline with Stop Word Remover as the final step
pipeline = Pipeline([
    ('tokenizer', NepaliLexicalAnalyzer()), 
    ('lemmatizer', NepaliHMMLemmatizer()),
    ('stop_word_remover', NepaliStopWordRemover(NEPALI_STOP_WORDS))
])

# Fit Pipeline directly on raw sentences and target lemmas
pipeline.fit(X_train_raw, y_train_lemmas)

# Save the complete pipeline
joblib.dump(pipeline, 'nepali_hmm_pipeline.pkl')
print("Model trained and saved successfully.")

Model trained and saved successfully.


## load the saved model and test

In [3]:

import joblib

# Load the previously saved pipeline from disk
loaded_pipeline = joblib.load('nepali_hmm_pipeline.pkl')

# Test on a longer, unseen sentence with vocabulary not present in training data
new_test_data = [
    "विद्यार्थीहरुले पुस्तकालयमा नयाँ किताबहरु पढे। शिक्षकहरुले कक्षाकोठामा राम्रो पाठ पढाए र विद्यार्थीहरु धेरै खुसी भए।"
]
new_processed_output = loaded_pipeline.transform(new_test_data)
print("Loaded Pipeline Output:", new_processed_output)

Loaded Pipeline Output: [['विद्यार्थी', 'पुस्तकालयमा', 'नयाँ', 'किताब', 'पढ्नु', 'शिक्षकहरु', 'कक्षाकोठा', 'राम्रो', 'पाठ', 'पढाए', 'विद्यार्थी', 'खुसी', 'हुनु']]
